# Portfolio Analytics

Analyses everything the desk holds: HL perps and spot in the configured wallets, the same wallets' tokens on other chains (staked VVV counts as VVV; stablecoins are cash), and IB stock positions read through TWS. Assets split into two groups, each measured against its own benchmark on its own calendar: **Crypto vs BTC** over every day, and **TradFi vs SPY** over US trading days (IB stocks and HL HIP-3 stock, index, commodity and FX perps). Prices come from HL candles, CoinGecko, DefiLlama (wallet tokens, by contract) and IB (adjusted closes, and SPY). TWS must be running for IB positions; without it they are skipped and saved prices are used where present.

The notebook then optimizes a constrained long/short Kelly portfolio, compares it with BTC and SPY, fits a factor model, and saves the results. It never places trades.

## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [ ]:
import sys
from importlib import reload
sys.path.insert(0, '..')

import factor_model, portfolio_risk, desk.hl.book_exposure
import portfolio_analytics_implementation
# Pick up code changes without restarting the kernel: helpers first, then the module that imports them.
for module in (factor_model, portfolio_risk, desk.hl.book_exposure, portfolio_analytics_implementation):
    reload(module)
from portfolio_analytics_implementation import PortfolioAnalytics

WALLET_ADDRESSES = [
    '0x...c248',
    '0x...07f2',
]

# Assets with less price history stay in the benchmark table but not in Kelly or factor sections.
MIN_HISTORY_DAYS = 180
# Years of daily prices to build (CoinGecko gives one; HL candles and DefiLlama the rest).
HISTORY_YEARS = 5

analysis=PortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES, min_history_days=MIN_HISTORY_DAYS, history_years=HISTORY_YEARS)


## Highlights

What stands out, what needs attention, and what changed since the last run, across every section below. This cell reserves the space; the `show_highlights()` cell near the end fills it in once the sections have run (Run All does both).

In [ ]:
analysis.highlights_panel()

## Market data refresh

Discovers current nonzero perp positions and spot balances. Perp prices come from HL daily candles (including HIP-3 markets); CoinGecko supplies spot-only prices and market-cap/volume data without requiring an API key. Assets without enough common daily returns against BTC are reported and skipped. Edit `WALLET_ADDRESSES` above to select wallets.


In [ ]:
analysis.refresh_market_data()


## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [ ]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [ ]:
analysis.calculate_returns()


## Correlation heatmaps

One heatmap per group on its own calendar: Crypto (HL crypto perps and spot, wallet tokens) over every day, TradFi (IB stocks and HL HIP-3 stock, index, commodity and FX perps) over US trading days. When both are held, a third heatmap shows crypto against TradFi on trading days, with 24/7 prices sampled at each trading day's close.

In [ ]:
analysis.plot_correlations()


## Benchmark exposures

Each asset against its own benchmark: crypto against BTC (365-day annualization), TradFi against SPY (252-day). Correlation, beta with a 90% bootstrap range, alpha, volatility and Sharpe use each asset's full pairwise history, not just the dates every asset shares.

In [ ]:
analysis.show_benchmark_exposures()

## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [ ]:
analysis.optimize_portfolio()


## Portfolio performance and risk metrics

Charts the Kelly portfolio against BTC, and against SPY when TradFi assets are held. A book with TradFi assets uses US trading days and 252-day annualization; a crypto-only book uses every day and 365. Beta is reported to BTC and to SPY, alpha is net of both, and a sleeve table splits the book into its Crypto and TradFi parts.

In [ ]:
analysis.analyze_performance()


## Factor model

Fits exposures to BTC, BTC momentum (long or short BTC by the sign of its lagged 30-day return), BTC realized volatility (lagged 30-day), a funding spread, CoinGecko cross-sectional crypto factors where available, and SPY when TradFi assets are held. Crypto factors are built from daily data and compounded onto US trading days for a mixed book.

In [ ]:
analysis.fit_factor_model()


## Style-factor model and return attribution

The industry-standard multi-factor regression: the book's daily excess return = alpha + Σ beta × factor + residual. Equity styles from the Ken French data library (market, size, value, quality, investment, 12-1 month momentum, 1-month reversal, developed-ex-US domicile, tech sector) plus a dividend factor (SCHD − SPY), and the crypto block (BTC, alt market, BTC momentum, funding). Alpha is the intercept; specific return (alpha + residual) is the return attributable to selection. The attribution table and chart split the book's return into factor contributions and specific return. French data updates monthly, so the latest weeks join after its next release. Run after `fit_factor_model()`.

In [ ]:
analysis.fit_style_model()

## Style-factor attribution of the actual books

The same regression on what was actually traded, so the alpha here is the desk's real specific return. **HL book**: every configured wallet combined into one time-weighted index from the saved HL history, so deposits, withdrawals and transfers between wallets are not performance. HL keeps detailed history for about a month, so the model runs on weekly returns until 120 consecutive daily returns have been saved, then on daily. **IB book**: daily NAV and deposits from IB Flex once set up (see `desk/venues/ib_flex.py`), otherwise the desk's saved snapshots, which cannot separate deposits from performance. With few periods only the core factors are kept.

In [ ]:
analysis.fit_style_model_hl_book()

## HL book exposure and style betas by regime

A single beta over periods with opposite positions is a weighted average of exposures the book never held at once, so it is unusable for risk. This section rebuilds the HL book's **net and gross exposure per leg** (BTC, alts, TradFi) every week from fills, token transfers and today's balances, splits the history into **regimes** by the sign of each leg and whether gross exposure exceeds 1× equity, and then reports:

- **betas per regime** next to the pooled fit (the current regime is also fit on daily returns when it is too short for weekly betas);
- a **regime-interaction model** (separate intercept and betas per regime, pooled Newey-West errors) that tests whether each beta changed, once two regimes have enough weeks;
- **rolling 12-week betas**, so sign flips are visible instead of averaged away.

Regimes before the crypto factors begin (about a year of CoinGecko data) show exposure only. Run after `fit_factor_model()`.

In [ ]:
analysis.style_regimes_hl_book()

In [ ]:
analysis.fit_style_model_ib_book()

## Cross-sectional factor premia

Tests whether characteristics earn a premium across crypto assets, with t-stats that respect correlation across assets. **Fama-MacBeth** runs one cross-sectional regression per day of every asset's return on its standardized, one-day-lagged characteristic, then takes the t-stat from the time series of daily slopes (Newey-West), so a day when all assets move together moves only that day's slope. **Portfolio sorts** give the same test as a long top-half / short bottom-half spread. Characteristics: 30-day momentum, 30-day volatility and 60-day BTC beta from prices (every crypto asset, including short-history ones), size and turnover where CoinGecko has market data, and 7-day funding for perps. A joint regression controls the broadly covered characteristics for each other.

In [ ]:
analysis.estimate_factor_premia()

## Alpha split-sample test

Splits the factor-model dates in half. Out-of-sample rows fit the Kelly weights on one half and measure factor-adjusted alpha on the other, so the weights never see the dates they are judged on.


In [ ]:
analysis.test_alpha_out_of_sample()


## Fill in the highlights

Summarizes the sections above into the Highlights panel at the top of the notebook (shown here instead if the panel cell was not run).


In [ ]:
analysis.show_highlights()

## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [ ]:
analysis.save_outputs()
